# Autoencoder — PaySim Fraud Detection

**Goal:** Test an Autoencoder anomaly detector on the same PaySim setup used by the Isolation Forest notebook.

This notebook is intentionally detailed. Run **one cell at a time** and inspect the output before continuing.

### Fair comparison rules
- Same PaySim data path.
- Same random seed.
- Same feature engineering.
- Same 60/20/20 split.
- Same normal-only training principle.
- Threshold selected on validation data.
- Final metrics reported on the untouched test set.
- `isFraud` is never used as an Autoencoder input.

In [5]:
# BLOCK 1 — Imports and environment check

import os
import json
time_import = None
import time
import warnings
from pathlib import Path

# Suppress noisy oneDNN and TensorFlow C++ logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    pass

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

print("=" * 80)
print("BLOCK 1: ENVIRONMENT CHECK")
print("=" * 80)

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

try:
    import sklearn
    print("Scikit-learn version:", sklearn.__version__)
except Exception as exc:
    print("Could not read scikit-learn version:", exc)

try:
    import tensorflow as tf
    print("TensorFlow version :", tf.__version__)
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for this notebook. "
        "Install it in the same Python environment used by Jupyter, then restart the kernel."
    ) from exc

print("Environment is ready.")


SystemError: <built-in function isinstance> returned a result with an exception set

### Output after Block 1

You should see Pandas, NumPy, scikit-learn, and TensorFlow versions.

If TensorFlow is missing, install it in the Jupyter kernel environment before proceeding.

In [ ]:
# BLOCK 2 — Configuration

DATA_PATH = Path(r".\data\PS_20174392719_1491204439457_log.csv")

MAX_ROWS = 500_000

RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_FROM_REMAINDER = 0.25

EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
PATIENCE = 3

MODEL_DIR = Path("models")
RESULT_DIR = Path("results")
MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

print("=" * 80)
print("BLOCK 2: CONFIGURATION")
print("=" * 80)
print("Dataset path        :", DATA_PATH.resolve())
print("Maximum rows        :", MAX_ROWS)
print("Random state        :", RANDOM_STATE)
print("Train / Validation / Test target split: 60% / 20% / 20%")
print("Epochs              :", EPOCHS)
print("Batch size          :", BATCH_SIZE)
print("Learning rate       :", LEARNING_RATE)
print("Early-stop patience :", PATIENCE)

In [ ]:
# BLOCK 3 — Load PaySim data

print("=" * 80)
print("BLOCK 3: LOADING PAYSIM DATA")
print("=" * 80)

if not DATA_PATH.exists():
    fallback_path = Path.cwd() / "data" / "PS_20174392719_1491204439457_log.csv"
    if fallback_path.exists():
        DATA_PATH = fallback_path
    else:
        raise FileNotFoundError(
            f"PaySim file was not found at: {DATA_PATH.resolve()}\n"
            "Please ensure the CSV file is placed inside the 'data' directory."
        )

start = time.perf_counter()

df = pd.read_csv(
    DATA_PATH,
    nrows=MAX_ROWS,
    low_memory=False
)

load_time = time.perf_counter() - start

print(f"Rows loaded      : {len(df):,}")
print(f"Columns loaded   : {len(df.columns)}")
print(f"Load time        : {load_time:.2f} seconds")
print()
print("Columns:")
print(list(df.columns))
print()
display(df.head())


In [ ]:
# BLOCK 4 — Data quality and fraud distribution

print("=" * 80)
print("BLOCK 4: DATA QUALITY CHECK")
print("=" * 80)

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print()
print("Fraud distribution:")
label_counts = df["isFraud"].value_counts().sort_index()
label_pct = (df["isFraud"].value_counts(normalize=True).sort_index() * 100).round(4)

distribution = pd.DataFrame({
    "count": label_counts,
    "percentage": label_pct
})
display(distribution)

print(f"Fraud transactions : {int(label_counts.get(1, 0)):,}")
print(f"Legitimate txns    : {int(label_counts.get(0, 0)):,}")
print(f"Fraud rate         : {float(label_pct.get(1, 0)):.4f}%")

In [ ]:
# BLOCK 5 — Feature engineering (must match Isolation Forest notebook)

print("=" * 80)
print("BLOCK 5: FEATURE ENGINEERING")
print("=" * 80)

work = df.copy()

work["orig_balance_change"] = work["oldbalanceOrg"] - work["newbalanceOrig"]
work["dest_balance_change"] = work["newbalanceDest"] - work["oldbalanceDest"]
work["orig_balance_error"] = work["amount"] - work["orig_balance_change"]
work["dest_balance_error"] = work["amount"] - work["dest_balance_change"]

work["orig_zero_after"] = (work["newbalanceOrig"] == 0).astype(int)
work["dest_zero_before"] = (work["oldbalanceDest"] == 0).astype(int)
work["dest_zero_after"] = (work["newbalanceDest"] == 0).astype(int)

work["log_amount"] = np.log1p(work["amount"])

work = pd.get_dummies(work, columns=["type"], prefix="type", dtype=int)

# Ensure all 5 expected PaySim transaction types exist as dummy columns
for t in ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]:
    col = f"type_{t}"
    if col not in work.columns:
        work[col] = 0

feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "orig_balance_error",
    "dest_balance_error",
    "orig_zero_after",
    "dest_zero_before",
    "dest_zero_after",
    "log_amount",
]

type_columns = sorted([c for c in work.columns if c.startswith("type_")])
feature_columns += type_columns

X = work[feature_columns].astype(np.float32)
y = work["isFraud"].astype(int)

print("Number of features :", X.shape[1])
for i, name in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {name}")

print()
print("Feature matrix shape:", X.shape)
display(X.head())


In [ ]:
# BLOCK 6 — Train / validation / test split

print("=" * 80)
print("BLOCK 6: FAIR DATA SPLIT")
print("=" * 80)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VALIDATION_SIZE_FROM_REMAINDER,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print(f"Training rows   : {len(X_train):,}")
print(f"Validation rows : {len(X_val):,}")
print(f"Test rows       : {len(X_test):,}")

for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(
        f"{name:12s} -> fraud={int(labels.sum()):,}, "
        f"fraud_rate={labels.mean() * 100:.4f}%"
    )

normal_mask = y_train == 0
X_train_normal = X_train.loc[normal_mask].copy()

print()
print(f"Normal-only training rows: {len(X_train_normal):,}")

In [ ]:
# BLOCK 7 — Standardize the features

print("=" * 80)
print("BLOCK 7: FEATURE SCALING")
print("=" * 80)

scaler = StandardScaler()

X_train_normal_scaled = scaler.fit_transform(X_train_normal).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Train matrix :", X_train_normal_scaled.shape)
print("Val matrix   :", X_val_scaled.shape)
print("Test matrix  :", X_test_scaled.shape)
print()
print("Mean of first 5 scaled columns:")
print(np.round(X_train_normal_scaled.mean(axis=0)[:5], 6))
print("Std of first 5 scaled columns:")
print(np.round(X_train_normal_scaled.std(axis=0)[:5], 6))

### Output after Block 7

The Autoencoder is sensitive to feature scale, so standardization is essential.

The scaler is fitted on legitimate training data only.

In [ ]:
# BLOCK 8 — Build the Autoencoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(RANDOM_STATE)

input_dim = X_train_normal_scaled.shape[1]

inputs = keras.Input(shape=(input_dim,), name="transaction_features")

x = layers.Dense(32, activation="relu", name="encoder_dense_1")(inputs)
x = layers.Dense(16, activation="relu", name="encoder_dense_2")(x)
latent = layers.Dense(8, activation="relu", name="latent")(x)

x = layers.Dense(16, activation="relu", name="decoder_dense_1")(latent)
x = layers.Dense(32, activation="relu", name="decoder_dense_2")(x)
outputs = layers.Dense(input_dim, activation="linear", name="reconstruction")(x)

autoencoder = keras.Model(inputs, outputs, name="paysim_autoencoder")

autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

print("=" * 80)
print("BLOCK 8: AUTOENCODER ARCHITECTURE")
print("=" * 80)
autoencoder.summary()

### Output after Block 8

The model compresses the transaction feature vector into a small latent representation and then reconstructs it.

The key idea is:

```text
Normal transaction
      ↓
Encoder
      ↓
Latent representation
      ↓
Decoder
      ↓
Reconstructed transaction
```

Transactions that reconstruct poorly should receive a higher anomaly score.

In [ ]:
# BLOCK 9 — Train the Autoencoder on legitimate transactions

print("=" * 80)
print("BLOCK 9: TRAINING AUTOENCODER")
print("=" * 80)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = autoencoder.fit(
    X_train_normal_scaled,
    X_train_normal_scaled,
    validation_split=0.10,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=1,
    callbacks=callbacks
)

training_time = time.perf_counter() - start

print()
print(f"Training time        : {training_time:.2f} seconds")
print(f"Epochs actually run  : {len(history.history['loss'])}")
print(f"Best validation loss : {min(history.history['val_loss']):.8f}")

In [ ]:
# BLOCK 10 — Plot training history

print("=" * 80)
print("BLOCK 10: TRAINING CURVES")
print("=" * 80)

plt.figure(figsize=(9, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Autoencoder Training History")
plt.legend()
plt.show()

print("Use this graph to check whether training and validation reconstruction loss behave sensibly.")

In [ ]:
# BLOCK 11 — Calculate reconstruction error

print("=" * 80)
print("BLOCK 11: RECONSTRUCTION ERROR")
print("=" * 80)

def reconstruction_error(model, data):
    reconstructed = model.predict(data, batch_size=BATCH_SIZE, verbose=0)
    return np.mean(np.square(data - reconstructed), axis=1)

val_scores = reconstruction_error(autoencoder, X_val_scaled)
test_scores = reconstruction_error(autoencoder, X_test_scaled)

print("Validation reconstruction-error statistics:")
print(pd.Series(val_scores).describe())

print()
print("Test reconstruction-error statistics:")
print(pd.Series(test_scores).describe())

print()
print("Score direction:")
print("Higher reconstruction error => more anomalous / more suspicious")

### Output after Block 11

This is the Autoencoder equivalent of the Isolation Forest anomaly score.

For every transaction, we calculate the mean squared reconstruction error.

In [ ]:
# BLOCK 12 — Select threshold on validation data

print("=" * 80)
print("BLOCK 12: VALIDATION THRESHOLD SELECTION")
print("=" * 80)

thresholds = np.quantile(val_scores, np.linspace(0.90, 0.9999, 300))

best_threshold = None
best_f1 = -1.0
best_precision = 0.0
best_recall = 0.0

for threshold in thresholds:
    val_pred = (val_scores >= threshold).astype(int)

    p = precision_score(y_val, val_pred, zero_division=0)
    r = recall_score(y_val, val_pred, zero_division=0)
    f = f1_score(y_val, val_pred, zero_division=0)

    if f > best_f1:
        best_f1 = f
        best_threshold = float(threshold)
        best_precision = float(p)
        best_recall = float(r)

if best_threshold is None:
    best_threshold = float(np.percentile(val_scores, 95))

print(f"Selected threshold  : {best_threshold:.8f}")
print(f"Validation Precision : {best_precision:.4f}")
print(f"Validation Recall    : {best_recall:.4f}")
print(f"Validation F1        : {best_f1:.4f}")


In [ ]:
# BLOCK 13 — Final test evaluation

print("=" * 80)
print("BLOCK 13: FINAL AUTOENCODER EVALUATION")
print("=" * 80)

test_pred = (test_scores >= best_threshold).astype(int)

precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, test_scores)
pr_auc = average_precision_score(y_test, test_scores)
cm = confusion_matrix(y_test, test_pred)

tn, fp, fn, tp = cm.ravel()

print(f"Precision            : {precision:.4f}")
print(f"Recall               : {recall:.4f}")
print(f"F1-score             : {f1:.4f}")
print(f"ROC-AUC              : {roc_auc:.4f}")
print(f"PR-AUC               : {pr_auc:.4f}")
print()
print(f"True Negatives (TN)  : {tn:,}")
print(f"False Positives (FP) : {fp:,}")
print(f"False Negatives (FN) : {fn:,}")
print(f"True Positives (TP)  : {tp:,}")
print()
print("Classification report:")
print(classification_report(y_test, test_pred, digits=4, zero_division=0))

display(pd.DataFrame(
    cm,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
))

In [ ]:
# BLOCK 14 — Visual evaluation

print("=" * 80)
print("BLOCK 14: SCORE DISTRIBUTION")
print("=" * 80)

plt.figure(figsize=(9, 5))
plt.hist(test_scores[y_test.values == 0], bins=100, alpha=0.65, label="Legitimate")
plt.hist(test_scores[y_test.values == 1], bins=100, alpha=0.65, label="Fraud")
plt.axvline(best_threshold, linestyle="--", linewidth=2, label="Selected threshold")
plt.xlabel("Reconstruction error")
plt.ylabel("Number of transactions")
plt.title("Autoencoder — Test Reconstruction Error")
plt.legend()
plt.show()

print("The threshold line is the operating point used for final test predictions.")

In [ ]:
# BLOCK 15 — Save model, scaler, features, and results

import joblib

print("=" * 80)
print("BLOCK 15: SAVING ARTIFACTS")
print("=" * 80)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

autoencoder.save(str(MODEL_DIR / "autoencoder.keras"))
joblib.dump(scaler, str(MODEL_DIR / "autoencoder_scaler.pkl"))

with open(MODEL_DIR / "autoencoder_features.json", "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

results = {
    "model": "Autoencoder",
    "rows_loaded": int(len(df)),
    "n_features": int(X.shape[1]),
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "test_rows": int(len(X_test)),
    "normal_train_rows": int(len(X_train_normal)),
    "training_time_seconds": float(training_time),
    "epochs_run": int(len(history.history["loss"])),
    "threshold": float(best_threshold),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
}

with open(RESULT_DIR / "autoencoder_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved:")
print(" - models/autoencoder.keras")
print(" - models/autoencoder_scaler.pkl")
print(" - models/autoencoder_features.json")
print(" - results/autoencoder_results.json")
print()
print("Autoencoder experiment complete.")


## Final note

Do not choose the final model until both notebooks have been executed.

The next comparison should use the two saved result files and examine:
- Precision
- Recall
- F1-score
- PR-AUC
- False Negatives
- False Positives
- Training time
- Inference latency

Then we can test the shortlisted model inside the Kafka → Spark streaming path.